# Monte Carlo Test Analysis

This notebook analyzes the results of Monte Carlo tests for inheritance calculations.

In [ ]:
import json
import os
from collections import defaultdict
from pathlib import Path

import numpy as np

# Add the src directory to the path
import sys
sys.path.insert(0, os.path.join(os.getcwd(), "src"))

from farady import calculate_from_dict, _build_distribution

In [ ]:
# Categories for test cases
CATEGORIES = ("ordinary", "no_fare", "hawashi")

def load_test_cases():
    """Load test cases from the test_cases.npz file."""
    data_path = Path("tests/data/test_cases.npz")
    if not data_path.exists():
        raise FileNotFoundError(f"Test data not found at {data_path}")
    
    data = np.load(data_path, allow_pickle=True)
    return {cat: list(data[cat]) for cat in CATEGORIES}

def case_to_result(case_dict):
    """Run calculation on a case and extract key results."""
    case = calculate_from_dict(case_dict)
    
    # Extract heir shares
    numerators = {
        name: int(heir.get("shares", 0))
        for name, heir in zip(case._all_heir_names, case._all_heirs)
        if heir.get("shares")
    }
    
    # Build distribution
    distribution = _build_distribution(case)
    
    # Calculate total from distribution
    total = sum(item["fraction"] for item in distribution.values()) if distribution else 0
    
    return {
        "original_case": case_dict,
        "distribution": distribution,
        "ending": case.ending,
        "asib": case.asib,
        "total": total,
        "status": case.status,
        "raas": case.raas,
        "numerators": numerators,
    }

In [ ]:
# Load test cases
print("Loading test cases...")
cases = load_test_cases()

# Limit for analysis (default 1000 like in tests)
limit = int(os.environ.get("FARADY_MONTE_LIMIT", "100"))

print(f"Processing {limit} cases per category...")
results = {}
for cat in CATEGORIES:
    cat_cases = cases[cat][:limit] if limit > 0 else []
    results[cat] = [(c, case_to_result(c)) for c in cat_cases]
    print(f"  {cat}: {len(cat_cases)} cases processed")

In [ ]:
# Analyze distribution totals
print("=== TOTAL ANALYSIS ===")

for cat in CATEGORIES:
    cat_results = results[cat]
    totals = [r[1]["total"] for r in cat_results]
    
    print(f"\n{cat.upper()} category:")
    print(f"  Count: {len(totals)}")
    print(f"  Min: {min(totals):.4f}")
    print(f"  Max: {max(totals):.4f}")
    print(f"  Mean: {np.mean(totals):.4f}")
    print(f"  Std: {np.std(totals):.4f}")
    
    # Count how many are exactly 1.0
    exact_ones = sum(1 for t in totals if round(t, 2) == 1.0)
    print(f"  Exactly 1.0: {exact_ones} ({exact_ones/len(totals)*100:.1f}%)")
    
    # Count anomalies
    anomalies = [(i, r[0], t) for i, (r, t) in enumerate(zip(cat_results, totals)) if round(t, 2) != 1.0]
    print(f"  Anomalies: {len(anomalies)} ({len(anomalies)/len(totals)*100:.1f}%)")
    
    if anomalies:
        print("  Sample anomalies:")
        for i, (idx, case, total) in enumerate(anomalies[:5]):
            print(f"    {i+1}. Case {idx}: total={total:.4f}, case={case}")

In [ ]:
# Analyze ibn share allocation
print("\n=== IBN SHARE ANALYSIS ===")

for cat in CATEGORIES:
    cat_results = results[cat]
    # Cases where ibn is present
    ibn_cases = [(i, r) for i, r in enumerate(cat_results) if r[0].get("ibn", 0) > 0]
    # Cases where ibn is present but gets no share
    ibn_no_share = [
        (i, r) for i, r in ibn_cases 
        if r[1]["numerators"].get("ibn", 0) == 0
    ]
    
    print(f"\n{cat.upper()} category:")
    print(f"  Cases with ibn: {len(ibn_cases)}")
    print(f"  Cases with ibn but no share: {len(ibn_no_share)}")
    if ibn_cases:
        print(f"  Percentage with no share: {len(ibn_no_share)/len(ibn_cases)*100:.1f}%")
    
    if ibn_no_share:
        print("  Sample cases with ibn but no share:")
        for i, (idx, (case, result)) in enumerate(ibn_no_share[:5]):
            print(f"    {i+1}. Case {idx}: {case[0]}")

In [ ]:
# Analyze calculation endings
print("\n=== ENDING ANALYSIS ===")

for cat in CATEGORIES:
    cat_results = results[cat]
    endings = defaultdict(int)
    
    for _, result in cat_results:
        ending = result["ending"] or "none"
        endings[ending] += 1
        
    print(f"\n{cat.upper()} category:")
    for ending, count in sorted(endings.items()):
        print(f"  {ending}: {count} ({count/len(cat_results)*100:.1f}%)")

In [ ]:
# Load and analyze failure cases from tests
print("\n=== FAILURE ANALYSIS ===")

def load_failure_cases(test_name):
    """Load failure cases from JSON files."""
    output_dir = Path("tests/output")
    if not output_dir.exists():
        return {}
        
    failure_files = list(output_dir.glob(f"failures_{test_name}_*.json"))
    if not failure_files:
        return {}
        
    # Get the most recent file
    latest_file = sorted(failure_files)[-1]
    
    with open(latest_file) as f:
        return json.load(f)

failure_data = load_failure_cases("test_total_always_one")
if not failure_data:
    print("No failure data found.")
else:
    for cat in CATEGORIES:
        failures = failure_data.get(cat, [])
        if failures:
            print(f"\n{cat.upper()} failures in total test:")
            print(f"  Count: {len(failures)}")
            print("  Sample failures:")
            for i, failure in enumerate(failures[:5]):
                case = failure["case"]
                result = failure["result"]
                print(f"    {i+1}. Total={result['total']:.4f}, case={case}")